# RI #1: Chunking variants and contextual chunking

This notebook compares five ways to cut and label the same contracts, on the harness from notebook 01, and measures when contextual chunking pays for its model calls. Every arm is scored twice: with retrieval scoped to the question's own contract, as RI #1 runs by default, and with retrieval across all contracts, where the question names the contract it is about.

The result, measured once on one seed: scope decides everything. Across all 123 contracts, contextual chunking with the cheaper head-and-neighbourhood prompt cuts retrieval failure at `k=10` from 0.6473 to 0.1187, for 1,418 model calls. Inside one contract, no arm beats plain chunks by more than seven questions in 1,188. The prompt Anthropic published, which shows the model the whole document, does worse than the cheap prompt (0.3695) until it is told to name the parties, and then it matches (0.1288).

After working through it, you can:

- Build plain, hierarchical, header-labelled, and contextual chunks over one corpus and score them on the same judgments.
- Explain why scope decides whether a chunk label can help, and measure it with a document hit rate.
- Replay contexts written once on the Vector cluster, or regenerate them on your own model.
- Report the indexing cost of each arm in calls and tokens, and estimate what it costs on a paid model.

## 1. Setup and evaluation data

Run the cells in order with `RUN_EXPENSIVE = False` first. The corpus loader downloads and verifies the pinned ContractNLI test split. The chunking cells run locally in seconds. Contexts, document names, and the retrieval scores are read from the committed results through the shared result cache, so the default path needs no API key and loads no embedding model.

With the flag on, the notebook calls `CONTEXT_MODEL` through the Vector proxy at `REQUESTS_PER_MINUTE` or less, about 4,400 calls in all, and embeds five arms with the local `BAAI/bge-small-en-v1.5` model on CPU. Each response is cached on disk, so an interrupted run resumes where it stopped and a repeated run makes no calls.

The data is the whole ContractNLI test split: 123 non-disclosure agreements (NDAs), 1,418 chunks of 1,200 characters with 200 characters of overlap, and 17 fixed hypotheses per NDA. Of the 2,091 hypothesis and NDA pairs, 1,188 have human-annotated evidence spans and 903 are labelled `NotMentioned`. Retrieval metrics use the 1,188 answerable questions. The hypotheses are the same for every NDA, so a hypothesis on its own carries no sign of which contract it is about. See [CORPUS.md](CORPUS.md) for versions, digests, and attribution.

In [1]:
import hashlib
import json
from pathlib import Path

from aieng.advanced_rag.cache import CacheKey, TieredResultCache
from aieng.advanced_rag.cache import paths as cache_paths
from aieng.advanced_rag.corpora import contract_nli
from aieng.advanced_rag.pipeline.hierarchical import detect_headings, split_documents, to_chunks
from aieng.advanced_rag.retrieval import contextual
from aieng.advanced_rag.retrieval.contextual import ContextSet, ModelPrice, naming_queries, price_contexts


# Model calls and embedding ship off. With the flag off, every number below is
# read from the committed result of the measured run.
RUN_EXPENSIVE = False

# The model that writes contexts and document names. Change it to regenerate
# the contexts on your own model; the committed set was written by this one.
CONTEXT_MODEL = "Qwen3_6-35B-A3B"
REQUESTS_PER_MINUTE = 40
K = 10
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
CHUNK_CHARS, OVERLAP_CHARS = 1200, 200

data_path = contract_nli.fetch_contract_nli()
CORPUS_SHA256 = hashlib.sha256(data_path.read_bytes()).hexdigest()
corpus = contract_nli.build_slice(data_path, chunk_chars=CHUNK_CHARS, overlap_chars=OVERLAP_CHARS)
documents = {doc_id: text for doc_id, text, _ in contract_nli.load_contract_nli(data_path)}
source_names = {str(d["id"]): d["file_name"] for d in json.loads(data_path.read_text())["documents"]}
summary = corpus.summary()
assert summary == {
    "contracts": 123,
    "chunks": 1418,
    "queries": 2091,
    "answerable_queries": 1188,
    "null_queries": 903,
    "null_rate": 0.4319,
    "mean_gold_chunks": 1.94,
}
print(
    f"ContractNLI test: {summary['contracts']} NDAs, {summary['chunks']} chunks of {CHUNK_CHARS:,} characters, "
    f"{summary['queries']:,} questions ({summary['answerable_queries']:,} answerable, {summary['null_queries']} null)"
)

REPO = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "aieng-advanced-rag").is_dir())
RI = REPO / "implementations" / "regulatory_policy_qa"
cache = TieredResultCache(global_dir=RI / "outputs" / "baseline_cache")


def measured_result(key, compute):
    """Read a measured artifact, or compute it when the flag is on."""
    if RUN_EXPENSIVE:
        return cache.load_or_compute(key, compute, force_refresh=True).value
    cached = cache.get(key)
    if cached is None:
        raise FileNotFoundError(f"No stored result for {key.producer}; set RUN_EXPENSIVE = True to produce it")
    return cached.value


def text_sha(text):
    """Fingerprint a prompt template, so editing it invalidates its results."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

ContractNLI test: 123 NDAs, 1418 chunks of 1,200 characters, 2,091 questions (1,188 answerable, 903 null)


## 2. Five ways to chunk and label the same contracts

Each arm changes one thing against the plain control. The hierarchical split uses the headings `detect_headings` finds, so many parents are shorter than 3,000 characters.

| Arm | What the retriever ranks | What the reader receives | Model calls to index |
| --- | --- | --- | --- |
| Plain | 1,200-character windows | The same windows | 0 |
| Hierarchical | 600-character children inside heading-aligned parents of up to 3,000 characters | The parent of each retrieved child | 0 |
| Metadata header | Plain windows, each prefixed with the file name, the first 200 characters of the NDA, and the section heading above the chunk | The same windows | 0 |
| Contextual, head and neighbourhood | Plain windows, each prefixed with a model-written context. The prompt carries the first 3,000 characters of the NDA and 1,500 characters either side of the chunk | The same windows | 1 per chunk |
| Contextual, full document | Plain windows with a model-written context. The prompt carries the whole NDA, then the chunk, which is Anthropic's published prompt | The same windows | 1 per chunk |
| Follow-up: full document, named | As above, with the head prompt's instruction to name the document, the parties, and the section | The same windows | 1 per chunk |

The follow-up row was added after the first run, to test why the full-document arm lost to the cheaper prompt. It changes only the instruction.

The head-and-neighbourhood prompt is the one an earlier CUAD measurement used, and the one behind its verdict to skip contextual chunking on scoped corpora. The full-document prompt costs more tokens per call, but the document is the same prefix for every chunk of that NDA, so a provider with prompt caching bills most of it at a tenth of the input price.

The header and context change only what the embedder sees (`Chunk.embed_text`). The span and the text stay the same, so the relevance judgments, which come from overlap between chunks and gold evidence spans, are identical for the plain, header, and both contextual arms. The hierarchical arm hands the reader a different unit, so its judgments are rebuilt from the parents it returns.

In [2]:
plain = list(corpus.chunks)
headings = {doc_id: detect_headings(text) for doc_id, text in documents.items()}
headered = contextual.header_chunks(plain, documents, source_names=source_names, headings=headings)

hierarchy = split_documents(documents, parent_chars=3000, child_chars=600, overlap_chars=100)
children = to_chunks(hierarchy.children)
parents = to_chunks(hierarchy.parents)
parent_of = {child.node_id: child.parent_id for child in hierarchy.children}

shape = hierarchy.summary()
print(f"plain        {len(plain):>5} chunks, mean {sum(len(c.text) for c in plain) / len(plain):,.0f} characters")
print(
    f"hierarchical {shape['children']:>5} children of {shape['mean_child_chars']:,.0f} characters "
    f"under {shape['parents']} parents of {shape['mean_parent_chars']:,.0f}"
)
print(
    f"header       {len(headered):>5} chunks, header adds {sum(len(c.indexed_text) - len(c.text) for c in headered) / len(headered):,.0f} characters on average"
)
print()
print("One header, as the embedder sees it before the chunk text:")
print(headered[5].indexed_text[: headered[5].indexed_text.index(headered[5].text)].strip())

plain         1418 chunks, mean 1,156 characters
hierarchical  2936 children of 541 characters under 848 parents of 1,627
header        1418 chunks, header adds 294 characters on average

One header, as the embedder sees it before the chunk text:
Document: 064 19 Non Disclosure Agreement 2019
Opening: NON-DISCLOSURE AGREEMENT Required under JEA's Confidential Information Protection Program THIS NON-DISCLOSURE AGREEMENT ("Agreement") is entered into this ______day of __________ by and between JEA, l
Section: 2. Prohibition on Disclosure of Confidential Information.


## 3. Questions that name their contract

Scoped retrieval answers "where in this NDA is the clause", because the harness already knows which NDA the question is about. Unscoped retrieval also has to answer "which NDA", and a templated hypothesis such as "Receiving Party shall not disclose the fact that Agreement was agreed or negotiated" fits all 123 of them equally well.

A person asking about one contract names it. To build that question, the model reads the first 2,000 characters of each NDA once and writes the name a person would use for it, such as the parties or the organization that issued it. That is 123 calls, cached like the contexts. Each hypothesis is then prefixed with its NDA's name: `In <name>: <hypothesis>`. The judgments do not change.

Some NDAs are blank templates with no party named in their opening, and those get a generic name. That is realistic: a question cannot single out a document whose text never says what it is.

In [3]:
client = None
if RUN_EXPENSIVE:
    from aieng.advanced_rag.batch import ProxyBatch
    from aieng.advanced_rag.graphrag.proxy import RateLimiter, ResponseCache, govern
    from aieng.advanced_rag.models import choose_models, profile_for
    from aieng.advanced_rag.proxy import find_env_file, resolve_api_key, resolve_base_url
    from openai import OpenAI

    MODELS = choose_models(show=True)
    CONTEXT_MODEL = MODELS.extraction.model
    responses = ResponseCache(cache_paths.root() / "responses" / "ri1-chunking-variants.sqlite")
    if profile_for(CONTEXT_MODEL).cache_control:
        # LiteLLM drops Anthropic's cache breakpoint, so a family that caches
        # prompts keeps the governed OpenAI client and its prompt cache.
        client = govern(
            OpenAI(
                base_url=resolve_base_url(start=REPO), api_key=resolve_api_key(start=REPO), timeout=180, max_retries=0
            ),
            cache=responses,
            limiter=RateLimiter(per_minute=REQUESTS_PER_MINUTE),
        )
    else:
        # ProxyBatch sends each set as one infermesh batch under the model's rate
        # limit, caches each response on disk so a re-run costs nothing, and
        # turns the on-prem model's thinking off.
        client = ProxyBatch(
            CONTEXT_MODEL, cache=responses, rpm=REQUESTS_PER_MINUTE, timeout=180, env_path=find_env_file(REPO)
        )

NAMING = {"head_chars": 2000, "max_tokens": 40, "temperature": 0.0, "thinking": False}
names_key = CacheKey(
    "ri1",
    "document-names",
    CONTEXT_MODEL,
    {"corpus_sha256": CORPUS_SHA256, "prompt_sha": text_sha(contextual.NAME_SYSTEM + contextual.NAME_PROMPT), **NAMING},
)


def write_names():
    """Ask the model, once per NDA, for the name a person would use for it."""
    found, usage = contextual.name_documents(documents, client, model=CONTEXT_MODEL, head_chars=2000, max_tokens=40)
    return {"model": CONTEXT_MODEL, "names": found, "usage": usage}


named = measured_result(names_key, write_names)
names = named["names"]
answerable = corpus.answerable
named_queries = naming_queries(answerable, names)
print(f"names written by {named['model']}: {len(names)} of {len(documents)} NDAs, {named['usage']['calls']} calls")
print()
for query in (named_queries[0], named_queries[400], named_queries[900]):
    print(f"  {query.question[:150]}")

names written by Qwen3_6-35B-A3B: 123 of 123 NDAs, 123 calls

  In JEA's Non-Disclosure Agreement: Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.
  In Confidentiality and Non-Disclosure Agreement between KDOL and Authorized Person: Receiving Party may retain some Confidential Information even afte
  In the mutual NDA between Cyberlux Corporation and International Consolidated Technologies: Receiving Party may create a copy of some Confidential Inf


## 4. Contexts written once and replayed anywhere

A context set is plain JSON: one record per chunk identifier, labelled with the model and the prompt version that wrote it, with the prompt and completion tokens that call billed. The committed sets were written by `Qwen3_6-35B-A3B` on the Vector cluster, with thinking off, at no charge. The cache key names the model, the prompt version, a fingerprint of the prompt text, the corpus digest, and the chunk settings, so a set written by a different model or prompt is a different artifact and never overwrites this one.

With the flag off, the cell below replays the committed sets. To regenerate them on your own model, set `CONTEXT_MODEL` in the first cell to a model your key can call and turn the flag on. Your sets are stored in your local cache tier and the committed ones stay where they are, so you can compare the two.

Calls go document by document, so the chunks of one NDA reach the provider within a few seconds of each other. That ordering is what lets a 5-minute prompt cache serve every chunk after the first.

In [4]:
CONTEXT_SETTINGS = {"max_tokens": 160, "temperature": 0.0, "thinking": False}


def contexts_key(prompt_version):
    """Name every input that determines a context set."""
    template = {
        contextual.HEAD_NEIGHBOURHOOD: contextual.PROMPT,
        contextual.FULL_DOCUMENT: contextual.FULL_DOCUMENT_PREFIX + contextual.FULL_DOCUMENT_SUFFIX,
        contextual.FULL_DOCUMENT_NAMED: contextual.FULL_DOCUMENT_PREFIX + contextual.FULL_DOCUMENT_NAMED_SUFFIX,
    }[prompt_version]
    return CacheKey(
        "ri1",
        "chunk-contexts",
        CONTEXT_MODEL,
        {
            "corpus_sha256": CORPUS_SHA256,
            "chunk_chars": CHUNK_CHARS,
            "overlap_chars": OVERLAP_CHARS,
            "prompt_version": prompt_version,
            "prompt_sha": text_sha(contextual.SYSTEM + template),
            **CONTEXT_SETTINGS,
        },
    )


def write_contexts(prompt_version):
    """Write one context per chunk, document by document."""
    written = contextual.generate_contexts(
        documents, plain, client, model=CONTEXT_MODEL, prompt_version=prompt_version, max_tokens=160
    )
    return written.to_json()


context_sets = {
    version: ContextSet.from_json(measured_result(contexts_key(version), lambda v=version: write_contexts(v)))
    for version in contextual.PROMPT_VERSIONS
}
for version, written in context_sets.items():
    usage = written.usage()
    print(
        f"{version:<23} {written.model}: {usage['calls']:,} contexts, {usage['failures']} failed, "
        f"{usage['prompt_tokens']:,} prompt + {usage['completion_tokens']:,} completion tokens"
    )

example = plain[5]
print()
print("Chunk", example.chunk_id, "as each prompt situated it:")
for version, written in context_sets.items():
    print(f"  {version:<23} {written.records[example.chunk_id].context}")

head-neighbourhood-v1   Qwen3_6-35B-A3B: 1,418 contexts, 0 failed, 2,449,430 prompt + 62,540 completion tokens
full-document-v1        Qwen3_6-35B-A3B: 1,418 contexts, 0 failed, 4,868,251 prompt + 59,949 completion tokens
full-document-named-v1  Qwen3_6-35B-A3B: 1,418 contexts, 0 failed, 4,898,029 prompt + 64,005 completion tokens

Chunk 1::0005 as each prompt situated it:
  head-neighbourhood-v1   This excerpt is from Section 2, "Prohibition on Disclosure of Confidential Information," of a Non-Disclosure Agreement between JEA and a Recipient, detailing restrictions on sharing confidential data with representatives and third parties.
  full-document-v1        This chunk details the specific obligations of the parties to limit disclosure to authorized representatives, maintain confidentiality, and restrict the use of confidential information solely to the transaction.
  full-document-named-v1  This excerpt is from Section 2 of the Non-Disclosure Agreement between JEA and the Recipient, 

## 5. Scoped retrieval

Every arm uses the same dense retriever and embedding model as notebook 01. The table reports retrieval failure, MRR, and nDCG at `k=10`, and `doc@10`, the share of questions with at least one chunk of the right NDA in the top 10. Under scope that share is 1.0 by construction.

The right half repeats the comparison at the character budget the plain arm reads at `k=10`, using `budget_k` from notebook 01. Hierarchical parents are up to 3,000 characters long, so ten of them hand the reader about three times as much text as ten plain chunks. Read the hierarchical row at its budget-matched depth.

In [5]:
if RUN_EXPENSIVE:
    import numpy as np
    from aieng.advanced_rag.evaluation.metrics import retrieved_chars
    from aieng.advanced_rag.retrieval.embedders import SentenceTransformerEmbedder
    from aieng.advanced_rag.retrieval.index import build_index
    from aieng.advanced_rag.retrieval.variants import Arm, judgments, rank, score_arm


REPORT_CONFIG = {
    "corpus_sha256": CORPUS_SHA256,
    "chunk_chars": CHUNK_CHARS,
    "overlap_chars": OVERLAP_CHARS,
    "hierarchy": {"parent_chars": 3000, "child_chars": 600, "overlap_chars": 100, "headings": True},
    "header": {"head_chars": 200, "source_name": True, "section": True},
    "context_model": CONTEXT_MODEL,
    "contexts": {v: contexts_key(v).digest for v in context_sets},
    "names": names_key.digest,
    "query_template": contextual.QUERY_TEMPLATE,
    "embedder": DENSE_MODEL,
    "query_prompt": "bge default",
    "k": K,
}
# Two reader budgets: the text the plain arm returns at k=3 and at k=10.
TIGHT_AND_WIDE = (3, 10)
REPORT_CONFIG["budgets"] = [f"plain@{d}" for d in TIGHT_AND_WIDE]
report_key = CacheKey("ri1", "chunking-variants-report", "bge-small-en-v1.5", REPORT_CONFIG)


def measure_arms():
    """Embed every arm, rank scoped and unscoped, and score on the read units."""
    arms = [
        Arm("plain", plain),
        Arm("hierarchical", children, read=parents, read_of=parent_of),
        Arm("metadata header", headered),
        Arm("contextual, head + neighbourhood", context_sets[contextual.HEAD_NEIGHBOURHOOD].apply(plain)),
        Arm("contextual, full document", context_sets[contextual.FULL_DOCUMENT].apply(plain)),
        Arm("follow-up: full document, named", context_sets[contextual.FULL_DOCUMENT_NAMED].apply(plain)),
    ]
    embedder = SentenceTransformerEmbedder(DENSE_MODEL)
    vectors = {
        "templated": np.asarray(embedder.encode_queries([q.question for q in answerable]), dtype=np.float32),
        "named": np.asarray(embedder.encode_queries([q.question for q in named_queries]), dtype=np.float32),
    }
    settings = {
        "scoped": ("templated", True),
        "unscoped, named": ("named", False),
        "unscoped, templated": ("templated", False),
    }
    depth = 4 * K
    results = {setting: {} for setting in settings}
    budgets = {}
    for arm in arms:
        index, _ = build_index(embedder, arm.indexed)
        qrels = judgments(arm.read_units, answerable)
        for setting, (queries, scoped) in settings.items():
            if setting == "unscoped, templated" and arm.name != "plain":
                continue
            runs = rank(index, vectors[queries], answerable, scoped=scoped, depth=depth, read_of=arm.read_of)
            if arm.name == "plain":
                chars = {c.chunk_id: len(c.text) for c in plain}
                budgets[setting] = {f"plain@{d}": round(retrieved_chars(runs, chars, d)) for d in TIGHT_AND_WIDE}
            results[setting][arm.name] = score_arm(
                runs, qrels, arm.read_units, answerable, k=K, budgets=budgets[setting]
            )
    return {
        "results": results,
        "budget_chars": budgets,
        "embedder": embedder.describe(),
        "context_model": CONTEXT_MODEL,
    }


measured = measured_result(report_key, measure_arms)
results = measured["results"]
source = "recomputed" if RUN_EXPENSIVE else "committed measurement"
print(f"embedder {DENSE_MODEL}, contexts by {measured['context_model']}, k={K} ({source})")


def show(setting):
    """Print one setting's table at fixed depth and at the two character budgets."""
    tight, wide = measured["budget_chars"][setting].values()
    print(f"{setting}: {len(answerable):,} answerable questions")
    print(f"budgets: plain at k=3 reads {tight:,} characters, plain at k=10 reads {wide:,}")
    print(f"{'':<34}{'fixed k=10':<33}| {'tight budget':<20}| wide budget")
    print(
        f"{'arm':<34}{'fail':>7}{'MRR':>8}{'nDCG':>8}{'doc':>8}  |"
        f"{'k':>3}{'chars':>7}{'fail':>8}  |{'k':>3}{'chars':>8}{'fail':>8}{'MRR':>8}"
    )
    for arm, row in results[setting].items():
        fixed = row["fixed"]
        narrow, broad = (row["budgets"][label] for label in measured["budget_chars"][setting])
        print(
            f"{arm:<34}{fixed['failure']:>7.4f}{fixed['mrr']:>8.4f}{fixed['ndcg']:>8.4f}{fixed['document_hit']:>8.4f}  |"
            f"{narrow['k']:>3}{narrow['chars']:>7,}{narrow['failure']:>8.4f}  |"
            f"{broad['k']:>3}{broad['chars']:>8,}{broad['failure']:>8.4f}{broad['mrr']:>8.4f}"
        )


show("scoped")

embedder BAAI/bge-small-en-v1.5, contexts by Qwen3_6-35B-A3B, k=10 (committed measurement)
scoped: 1,188 answerable questions
budgets: plain at k=3 reads 3,564 characters, plain at k=10 reads 9,985
                                  fixed k=10                       | tight budget        | wide budget
arm                                  fail     MRR    nDCG     doc  |  k  chars    fail  |  k   chars    fail     MRR
plain                              0.0093  0.7286  0.7518  1.0000  |  3  3,564  0.1465  | 10   9,985  0.0093  0.7286
hierarchical                       0.0017  0.8056  0.8476  1.0000  |  1  3,014  0.3300  |  6   9,823  0.0135  0.8041
metadata header                    0.0210  0.6406  0.6885  1.0000  |  3  3,563  0.2113  | 10   9,992  0.0210  0.6406
contextual, head + neighbourhood   0.0143  0.7203  0.7505  1.0000  |  3  3,567  0.1406  | 10   9,994  0.0143  0.7203
contextual, full document          0.0126  0.7118  0.7455  1.0000  |  3  3,562  0.1625  | 10   9,982  0.0126  0.71

Under scope, `k=10` returns about 10,000 characters of an NDA that averages 11,218, so almost every arm finds a gold chunk and the failure column at `k=10` cannot separate them. The tight budget, the text plain chunks return at `k=3`, is the informative column.

At that budget no arm wins. The head-and-neighbourhood contexts fail on 0.1406 of questions against plain's 0.1465, a difference of seven questions, and the other two contextual arms are level with plain or worse. This matches the earlier CUAD verdict. When the candidate pool is one contract, a label that says which contract a chunk belongs to is the same on every candidate, so it cannot break a tie.

Two arms lose under scope:

- The metadata header fails on 0.2113 at the tight budget. Its 294 characters of header are the same for every chunk of an NDA, and they take a fifth of what the embedder sees. For it to win, the header would have to carry information that differs between chunks of one document. The cheapest test is a header holding the section heading alone.
- Hierarchical retrieval has the best MRR at `k=10` (0.8056), but an NDA splits into about seven parents against about 11 plain chunks, so a random order already ranks a gold parent higher. At one parent, 3,014 characters, it fails on 0.3300. For it to win, one parent would have to fit the budget a single plain chunk does. The cheapest test is `parent_chars=1200`, or scoring the answers the reader writes from parents against those it writes from chunks.

## 6. Unscoped retrieval with questions that name the contract

The same arms, ranked across all 1,418 chunks of the 123 NDAs, with the named questions. The last line is the floor: the plain arm with the templated hypotheses and no scope, which has no way to tell one NDA from another.

In [6]:
show("unscoped, named")
print()
floor = results["unscoped, templated"]["plain"]["fixed"]
print(
    f"plain, unscoped, templated questions: failure {floor['failure']:.4f}, "
    f"right NDA in the top {K} for {floor['document_hit']:.1%} of questions"
)

unscoped, named: 1,188 answerable questions
budgets: plain at k=3 reads 3,542 characters, plain at k=10 reads 11,880
                                  fixed k=10                       | tight budget        | wide budget
arm                                  fail     MRR    nDCG     doc  |  k  chars    fail  |  k   chars    fail     MRR
plain                              0.6473  0.2247  0.1822  0.8502  |  3  3,542  0.7441  | 10  11,880  0.6473  0.2247
hierarchical                       0.5732  0.2659  0.2847  0.8249  |  1  3,133  0.8022  |  4  11,193  0.6684  0.2515
metadata header                    0.1936  0.4402  0.4863  0.9327  |  3  3,528  0.4604  | 10  11,687  0.1936  0.4402
contextual, head + neighbourhood   0.1187  0.5442  0.5743  0.9638  |  3  3,568  0.3258  | 10  11,828  0.1187  0.5442
contextual, full document          0.3695  0.3843  0.3625  0.9301  |  3  3,554  0.5160  | 10  11,841  0.3695  0.3843
follow-up: full document, named    0.1288  0.5379  0.5646  0.9512  |  3  3,564

Without scope, the arms separate by a wide margin. Plain chunks fail on 0.6473 of the named questions at `k=10`, even though they put some chunk of the right NDA in the top 10 for 85% of them. The name at the front of the question matches the chunks where the parties are named, usually the opening, and the clause the question asks about is often elsewhere.

A label on every chunk fixes that, because the name then matches every chunk of the right NDA equally and the rest of the question decides among them:

- The metadata header, at no model cost, fails on 0.1936.
- The head-and-neighbourhood contexts fail on 0.1187, the lowest in the table, with the right NDA in the top 10 for 96% of questions.
- The full-document contexts fail on 0.3695. On this corpus Anthropic's prompt describes the clause and usually leaves out the parties. The follow-up arm adds the instruction to name them and fails on 0.1288, within 12 questions of the cheaper prompt.

The difference between the two full-document arms is the most useful finding here. Showing the model the whole document did not help. Asking it to name what retrieval needs did.

One caution applies to every model-written arm. The same model wrote the document names in the questions and the contexts on the chunks, so they may agree in wording more than a person's question would. The header arm quotes the raw opening of the NDA and still reaches 0.1936. The cheapest test of how much the shared wording is worth is to write the names with a different model.

The templated floor shows why naming the document is the precondition: with no name in the question, plain retrieval finds the right NDA in its top 10 for one question in ten.

## 7. Indexing cost per arm

The model-free arms cost nothing beyond embedding, which runs locally. The contextual arms cost one call per chunk. The token counts below are what `Qwen3_6-35B-A3B` reported for each call.

The price columns apply list prices for the cheapest model of each paid family to those token counts. They are estimates in three ways: the prices change, your team's proxy price can differ from list, and another model's tokenizer counts the same prompt differently. The cached column assumes Anthropic's prompt caching through the proxy, which the proxy has to forward. Confirm that it does before you rely on the cached figure.

In [7]:
# List prices per million tokens for the cheapest model of each paid family, as
# recorded in the project's model-access design. They are estimates: check the
# provider's pricing page, and the price the proxy reports for your key, before
# you budget against them. Anthropic bills a 5-minute cache write at 1.25x input
# and a read at 0.1x, with a 512-token minimum prefix on Haiku 5.5. The Gemini and
# OpenAI rows bill every prompt token at the input price, which is an upper bound
# because both providers discount repeated prefixes on their own.
PRICES = [
    ModelPrice(
        "claude-haiku-5-5",
        0.10,
        0.50,
        cache_write_multiplier=1.25,
        cache_read_multiplier=0.10,
        min_cacheable_tokens=512,
    ),
    ModelPrice("gemini-3.1-flash-lite-preview", 0.25, 1.50),
    ModelPrice("gpt-5.4-nano", 0.20, 1.25),
]

model_free = {"plain": len(plain), "hierarchical": len(children), "metadata header": len(headered)}
print(f"indexing cost per arm, {summary['contracts']} NDAs ({CONTEXT_MODEL} tokens for the model-written arms)")
print(f"{'arm':<34}{'units':>7}{'calls':>7}{'prompt tok':>12}{'compl. tok':>11}{'tok/chunk':>10}")
for arm, units in model_free.items():
    print(f"{arm:<34}{units:>7,}{0:>7}{0:>12}{0:>11}{0:>10}")
labels = {
    contextual.HEAD_NEIGHBOURHOOD: "contextual, head + neighbourhood",
    contextual.FULL_DOCUMENT: "contextual, full document",
    contextual.FULL_DOCUMENT_NAMED: "follow-up: full document, named",
}
for version, written in context_sets.items():
    usage = written.usage()
    per_chunk = (usage["prompt_tokens"] + usage["completion_tokens"]) / usage["calls"]
    print(
        f"{labels[version]:<34}{len(plain):>7,}{usage['calls']:>7,}{usage['prompt_tokens']:>12,}"
        f"{usage['completion_tokens']:>11,}{per_chunk:>10,.0f}"
    )
print(
    f"{'document names (unscoped questions)':<34}{len(names):>7}{named['usage']['calls']:>7}"
    f"{named['usage']['prompt_tokens']:>12,}{named['usage']['completion_tokens']:>11,}"
)

print()
print("estimated price at list, US$ (estimates; token counts are the on-prem model's)")
print(f"{'model':<31}{'head + nbhd':>12}{'full doc':>10}{'cached':>8}{'full named':>12}{'cached':>8}")
for price in PRICES:
    row = f"{price.model:<31}"
    for version, width in (
        (contextual.HEAD_NEIGHBOURHOOD, 12),
        (contextual.FULL_DOCUMENT, 10),
        (contextual.FULL_DOCUMENT_NAMED, 12),
    ):
        row += f"{price_contexts(context_sets[version], price)['usd']:>{width}.2f}"
        if version != contextual.HEAD_NEIGHBOURHOOD:
            cached = price_contexts(context_sets[version], price, cached=True)["usd"] if price.caches else None
            row += f"{cached:>8.2f}" if cached is not None else f"{'n/a':>8}"
    print(row)
print("n/a: cached rates for this provider are not recorded here; the uncached figure is an upper bound")

reported = context_sets[contextual.FULL_DOCUMENT].usage()["cached_tokens_reported"]
total = context_sets[contextual.FULL_DOCUMENT].usage()["prompt_tokens"]
print()
print(
    f"the on-prem server reported {reported:,} of {total:,} full-document prompt tokens ({reported / total:.0%}) as prefix-cache reads"
)

indexing cost per arm, 123 NDAs (Qwen3_6-35B-A3B tokens for the model-written arms)
arm                                 units  calls  prompt tok compl. tok tok/chunk
plain                               1,418      0           0          0         0
hierarchical                        2,936      0           0          0         0
metadata header                     1,418      0           0          0         0
contextual, head + neighbourhood    1,418  1,418   2,449,430     62,540     1,771
contextual, full document           1,418  1,418   4,868,251     59,949     3,475
follow-up: full document, named     1,418  1,418   4,898,029     64,005     3,499
document names (unscoped questions)    123    123      65,644      1,786

estimated price at list, US$ (estimates; token counts are the on-prem model's)
model                           head + nbhd  full doc  cached  full named  cached
claude-haiku-5-5                       0.28      0.52    0.15        0.52    0.15
gemini-3.1-flash-lite-pre

The head-and-neighbourhood prompt costs 1,771 tokens per chunk and the full-document prompts about 3,500. Without caching, writing the head-and-neighbourhood contexts for all 123 NDAs is an estimated US$0.28 on `claude-haiku-5-5`, and the full-document set is US$0.52. With a 5-minute prompt cache, each NDA's document is written to the cache once and read for every later chunk, which brings the full-document set to an estimated US$0.15. The head prompt's 3,000-character opening is also a shared prefix, so it caches too, though the estimate above does not apply that.

At these prices, cost does not decide between the contextual prompts on a corpus this size. The instruction does. Scale the figures by your corpus: the estimate is linear in the number of chunks.

The on-prem server cached prefixes on its own, and reported 78% of the full-document prompt tokens as cache reads. Self-hosted serving gives the full-document prompt the same discount a paid provider's cache does.

## 8. The same method on CUAD and ServerFault

A separate run measured the head-and-neighbourhood prompt on two other corpora. Its committed numbers are below. Both runs used plain 1,200-character windows, the same embedder, and the embedding half of the method only, with no contextual BM25 and no reranker.

In [8]:
contrast = json.loads((RI / "outputs" / "contextual_contrast_cuad_serverfault.json").read_text())
produced = contrast["produced_by"]
print(f"source: {produced['source']}")
print(f"contexts by {produced['model']}, prompt {contrast['prompt_version']}, embedder {produced['embedder']}")
print()
print(
    f"{'corpus':<13}{'scope':<14}{'queries':>8}{'plain fail':>12}{'ctx fail':>10}{'plain MRR':>11}{'ctx MRR':>9}{'tokens':>12}"
)
for corpus_name, row in contrast["corpora"].items():
    tokens = row["cost"]["prompt_tokens"] + row["cost"]["completion_tokens"]
    print(
        f"{corpus_name:<13}{row['scope']:<14}{row['queries']:>8}"
        f"{row['plain']['retrieval_failure_rate']:>12.4f}{row['contextual']['retrieval_failure_rate']:>10.4f}"
        f"{row['plain']['mrr']:>11.4f}{row['contextual']['mrr']:>9.4f}{tokens:>12,}"
    )

source: contextual chunking runs on CUAD and ServerFault, made with code not included in this repository
contexts by Qwen3_6-35B-A3B, prompt head-neighbourhood-v1, embedder BAAI/bge-small-en-v1.5

corpus       scope          queries  plain fail  ctx fail  plain MRR  ctx MRR      tokens
CUAD         per document       250      0.2400    0.2640     0.4348   0.4400   2,089,867
ServerFault  corpus-wide        600      0.0133    0.0017     0.9238   0.9591   2,262,662


The two corpora point the same way as the ContractNLI tables. On CUAD, with retrieval scoped to one contract, contextual chunking raised failure from 0.2400 to 0.2640, which is six questions of 250, while MRR rose slightly. On ServerFault, where retrieval runs over the whole corpus, failure fell from 0.0133 to 0.0017. The ServerFault query is the post's title, and the written context usually repeats the title, so that figure is an upper bound on the gain. The ContractNLI unscoped run is a cleaner test of the same effect, because its questions name the document in words written separately from the contexts.

## 9. Which arm wins, and under which scope

Under unscoped retrieval with questions that name their contract, the measured winner is contextual chunking with the head-and-neighbourhood prompt. It cuts failure at `k=10` from 0.6473 to 0.1187, so 628 more of the 1,188 answerable questions find a gold chunk, at 1,418 calls and 2.5 million tokens on the on-prem model. The free metadata header gets most of the way, to 0.1936. The full-document prompt matches the winner only when it is told to name the parties.

Under single-contract scope there is no winner. Plain chunks are within seven questions of the best arm at the tight budget, so the extra calls buy nothing there.

These are single runs with no interval, on one embedding model, the embedding half of the method only. The published pipeline adds contextual BM25 and a reranker, and neither is measured here. Treat the scoped null as a result about this corpus and this retriever: contracts short enough that ten chunks cover most of one, scored on retrieval rather than on answers.

The practical rule for your project: if your questions go to a corpus without a filter that picks the document, label every chunk with the document it belongs to. Try the free header first, and pay for contexts when the header leaves a gap you can measure. If your system already knows which document a question is about, spend the budget elsewhere.

## 10. Run it on your own corpus and model

To regenerate the contexts on a paid model:

1. Set `CONTEXT_MODEL` in the first cell to a model your key can call through the proxy.
2. Set `RUN_EXPENSIVE = True`.
3. Run all cells.

The cost cell prints the calls and tokens your model reported, so you can compare them with the estimate above. For Anthropic models, `contextual.generate_contexts(..., cache_control=True)` marks the document as a cacheable prefix. Check `cached_tokens_reported` in the context set's usage to see whether the cache served it.

To apply the comparison to your own documents, build a `Slice` with character-offset judgments as in notebook 01, and pass your documents and chunks to the same functions. The functions live in `aieng.advanced_rag.retrieval.contextual` (prompts, context sets, headers, names, and the price estimate) and `aieng.advanced_rag.retrieval.variants` (scoped and unscoped ranking and the scores). For setup and repository navigation, continue with [GUIDE.md](GUIDE.md).